# 7장 실습: 텍스트로 지식 그래프 구축과 벡터 RAG

실습 가이드 7장 7-1 ~ 7-7의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | 빈 DB (AuraDB 또는 Docker의 빈 컨테이너) |
| .env 설정 | 빈 DB의 NEO4J 네 줄 |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

이 노트북은 DB에 데이터를 씁니다. 다른 데이터가 있는 DB에 연결하면 결과가 섞입니다. 다시 실행할 때는 먼저 DB를 비웁니다(아래 `RESET_DB`).

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [1]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 연결하고 레이블별 노드 수를 출력한다. expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# NEO4J는 접속 정보 묶음이다. 다른 LangChain 객체를 만들 때 **NEO4J로 그대로 넘긴다(실습 가이드 6-2).
# 연결 오류는 이 줄에서 난다. ServiceUnavailable: 서버가 꺼져 있거나 시작 중 / AuthError: 비밀번호 / APOC 오류: 플러그인 없음
# forbid: Recommendations(Movie)나 POLE(Crime)에 연결됐으면 여기서 오류로 멈춘다. 이 노트북은 DB를 지우므로 빈 DB에서만 실행한다.
graph, NEO4J = connect_graph(expect=[], forbid=["Movie", "Crime"])

연결: bolt://localhost:7689 | DB: neo4j


DB 비우기 (실습 가이드 7-1)

위 출력에 노드가 있으면 이 실습용 DB인지 확인한 뒤 `RESET_DB = True`로 바꿔 실행합니다. 모든 노드와 관계가 지워지고 되돌릴 수 없습니다.

In [2]:
# # True로 바꾸면 모든 노드와 관계를 지운다. 되돌릴 수 없으니 연결 셀 출력에서 실습용 빈 DB인지 먼저 확인한다.
# RESET_DB = False

# if RESET_DB:
#     # DETACH DELETE는 관계를 먼저 끊고 노드를 지운다. DELETE만 쓰면 관계가 남은 노드에서 오류가 난다.
#     graph.query("MATCH (n) DETACH DELETE n")
# print("노드 수:", graph.query("MATCH (n) RETURN count(n) AS n")[0]["n"])

## 1. 임베딩 원리: OpenAI API 직접 호출 (실습 가이드 7-3)

LangChain을 쓰기 전에 임베딩이 무엇을 돌려주는지 확인합니다. 같은 뜻의 한국어 문장과 영어 문장을 임베딩해 코사인 유사도를 계산합니다.

In [3]:
import numpy as np
from openai import OpenAI

# OPENAI_API_KEY는 연결 셀의 load_dotenv()로 이미 환경변수에 들어 있다.
client = OpenAI()
# 문장 여러 개를 리스트로 넘기면 한 번의 호출로 문장마다 벡터를 하나씩 받는다.
resp = client.embeddings.create(
    model="text-embedding-3-small",
    input=["마리 퀴리는 노벨상을 두 번 받았다.", "Marie Curie won two Nobel Prizes."],
)
v1, v2 = resp.data[0].embedding, resp.data[1].embedding
print(len(v1))   # 1536

# 코사인 유사도: 두 벡터의 방향이 같을수록 1에 가깝다. 언어가 달라도 뜻이 같으면 높게 나온다.
cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
print(round(cos, 3))

1536
0.188


## 2. 청크 분할 (실습 가이드 7-4)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 실습용 짧은 영어 원문. 실제로는 문서 파일을 읽어 넣는다.
sample_text = """
Marie Curie, 7 November 1867 - 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.
She was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.
Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.
She was, in 1906, the first woman to become a professor at the University of Paris.
"""

# chunk_size: 청크 하나의 최대 글자 수, chunk_overlap: 앞 청크와 겹치는 글자 수(잘린 경계의 문맥을 보존).
# 문단 → 줄 → 공백 순으로 자를 곳을 찾으므로 단어 중간에서 끊기지 않는다.
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
# Document 객체 리스트를 돌려준다. LLMGraphTransformer와 벡터 저장소가 모두 이 형식을 받는다.
documents = splitter.create_documents([sample_text])
print(len(documents))
for d in documents:
    print("-", d.page_content)

3
- Marie Curie, 7 November 1867 - 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.
- She was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.
- Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.
She was, in 1906, the first woman to become a professor at the University of Paris.


## 3. 엔티티·관계 추출과 적재 (실습 가이드 7-5)

`LLMGraphTransformer`가 청크마다 gpt-4o를 호출해 노드와 관계를 추출합니다. `allowed_nodes`, `allowed_relationships`로 허용할 이름을 정해 두면 실행마다 이름이 달라지는 문제를 줄일 수 있습니다.

In [5]:
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

# temperature=0: 같은 입력에 최대한 같은 결과. 추출 작업은 다양성보다 일관성이 중요하다.
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 허용 목록을 주면 LLM이 이 레이블과 관계 유형만 쓴다. 없으면 실행마다 이름이 제각각이 된다(과제 1에서 비교).
transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Organization", "Award", "Country", "Field"],
    allowed_relationships=["MARRIED_TO", "WON", "WORKED_AT", "CITIZEN_OF", "RESEARCHED"],
)
# 청크마다 LLM을 한 번씩 호출해 노드와 관계를 뽑는다. 청크 수만큼 시간과 비용이 든다.
graph_documents = transformer.convert_to_graph_documents(documents)

# 아직 DB에 쓰지 않았다. 추출 결과를 먼저 눈으로 확인한다.
for gd in graph_documents:
    print(gd.nodes)
    print(gd.relationships)

C:\Users\barne\AppData\Local\Temp\ipykernel_9276\80842712.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


[Node(id='Marie Curie', type='Person', properties={}), Node(id='Poland', type='Country', properties={}), Node(id='France', type='Country', properties={}), Node(id='Physics', type='Field', properties={}), Node(id='Chemistry', type='Field', properties={})]
[Relationship(source=Node(id='Marie Curie', type='Person', properties={}), target=Node(id='Poland', type='Country', properties={}), type='CITIZEN_OF', properties={}), Relationship(source=Node(id='Marie Curie', type='Person', properties={}), target=Node(id='France', type='Country', properties={}), type='CITIZEN_OF', properties={}), Relationship(source=Node(id='Marie Curie', type='Person', properties={}), target=Node(id='Physics', type='Field', properties={}), type='RESEARCHED', properties={}), Relationship(source=Node(id='Marie Curie', type='Person', properties={}), target=Node(id='Chemistry', type='Field', properties={}), type='RESEARCHED', properties={})]
[Node(id='Nobel Prize', type='Award', properties={}), Node(id='Scientific Fields

In [6]:
# baseEntityLabel=True: 모든 엔티티에 __Entity__ 레이블을 함께 붙인다(레이블과 상관없이 엔티티 전체를 찾을 때 쓴다).
# include_source=True: 원문 청크를 Document 노드로 저장하고 엔티티와 MENTIONS로 잇는다. 4절 벡터 검색이 이 Document를 쓴다.
# 노드는 MERGE로 쓰므로 같은 id의 엔티티는 하나로 합쳐진다.
graph.add_graph_documents(graph_documents, baseEntityLabel=True, include_source=True)

# 적재 결과: 원문 청크(Document)와 엔티티가 MENTIONS로 연결된다.
for row in graph.query("""
MATCH (n) RETURN labels(n) AS labels, count(*) AS n ORDER BY n DESC
"""):
    print(row)

{'labels': ['Document'], 'n': 3}
{'labels': ['__Entity__', 'Person'], 'n': 3}
{'labels': ['__Entity__', 'Field'], 'n': 3}
{'labels': ['__Entity__', 'Country'], 'n': 2}
{'labels': ['__Entity__', 'Organization'], 'n': 2}
{'labels': ['__Entity__', 'Award'], 'n': 1}


Neo4j Browser에서도 확인합니다.

```cypher
MATCH (d:Document)-[:MENTIONS]->(e)-[r]->(e2)
RETURN d, e, r, e2 LIMIT 50
```

## 4. 벡터 인덱스와 retriever (실습 가이드 7-6)

`Neo4jVector.from_existing_graph`는 `Document` 노드 중 임베딩이 없는 노드를 찾아 임베딩하고 벡터 인덱스까지 만듭니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

# 1절과 같은 모델. 문서와 질문은 반드시 같은 모델로 임베딩해야 비교할 수 있다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# DB에 이미 있는 Document 노드를 벡터 저장소로 만든다.
#   1) document_index 벡터 인덱스가 없으면 만든다.
#   2) embedding 속성이 없는 Document의 text를 임베딩해 embedding 속성에 저장한다.
vector_store = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    **NEO4J,
    index_name="document_index",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

# k=3: 질문과 가장 가까운 청크 3개를 돌려준다.
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 질문은 한국어, 문서는 영어지만 뜻으로 검색하므로 찾아진다.
for doc in retriever.invoke("마리 퀴리가 노벨상을 받은 이유는?"):
    print(doc.page_content)
    print("---")


text: She was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.
---

text: Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.
She was, in 1906, the first woman to become a professor at the University of Paris.
---

text: Marie Curie, 7 November 1867 - 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.
---


## 5. LCEL RAG 체인 (실습 가이드 7-7)

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# system: 답변 규칙(문서에만 근거, 없으면 모른다고 답함) / human: 실행할 때 채울 {context}, {question}
prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 문서만 근거로 한국어로 간결하게 답한다. 문서에 없으면 '정보가 부족합니다'라고 답한다."),
    ("human", "문서:\n{context}\n\n질문: {question}"),
])

# retriever는 Document 리스트를 돌려준다. 프롬프트에 넣으려면 문자열 하나로 합쳐야 한다.
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# LCEL 체인: | 왼쪽의 출력이 오른쪽의 입력이 된다.
#   질문 → {context: 검색 결과 문자열, question: 질문 그대로} → 프롬프트 → LLM → 문자열
# 딕셔너리의 각 값은 같은 질문을 받아 따로 실행된다. RunnablePassthrough()는 입력을 그대로 넘긴다.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print(rag_chain.invoke("마리 퀴리가 노벨상을 받은 이유는 무엇인가요?"))

정보가 부족합니다.


## 실습 과제

위의 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제, 정답 코드, 해설 순서로 되어 있습니다. 먼저 직접 작성해 본 뒤 정답 코드와 비교합니다.

### 과제 1. 허용 목록 없이 추출하기

`allowed_nodes`, `allowed_relationships`를 지운 `LLMGraphTransformer`로 같은 청크를 다시 추출하고, 3번 셀의 결과와 레이블·관계 이름을 비교하세요. 이 과제는 DB에 쓰지 않고 Python에서만 비교합니다(과제 2, 3에서 지금 DB를 계속 씁니다).

In [9]:
# 과제 1 정답
# allowed_nodes, allowed_relationships 없이 만든다. LLM이 레이블과 관계 이름을 스스로 정한다.
free_transformer = LLMGraphTransformer(llm=llm)
free_documents = free_transformer.convert_to_graph_documents(documents)

# 두 결과를 같은 기준(노드 수, 레이블 종류, 관계 종류)으로 비교하기 위한 요약 함수
def summarize(graph_docs):
    labels = sorted({n.type for gd in graph_docs for n in gd.nodes})
    rels = sorted({r.type for gd in graph_docs for r in gd.relationships})
    nodes = sum(len(gd.nodes) for gd in graph_docs)
    return nodes, labels, rels

for name, gds in [("허용 목록 있음", graph_documents), ("허용 목록 없음", free_documents)]:
    n, labels, rels = summarize(gds)
    print(f"== {name}: 노드 {n}개")
    print("  레이블:", labels)
    print("  관계:", rels)

== 허용 목록 있음: 노드 12개
  레이블: ['Award', 'Country', 'Field', 'Organization', 'Person']
  관계: ['CITIZEN_OF', 'MARRIED_TO', 'RESEARCHED', 'WON', 'WORKED_AT']
== 허용 목록 없음: 노드 16개
  레이블: ['Award', 'Date', 'Family', 'Field', 'Nationality', 'Organization', 'Person', 'Profession']
  관계: ['BIRTH_DATE', 'CO-WINNER', 'DEATH_DATE', 'HAS_NATIONALITY', 'HAS_PROFESSION', 'LEGACY', 'PROFESSOR', 'RESEARCHED', 'SPOUSE', 'WINNER', 'WON', 'WON_IN_TWO', 'WON_TWICE']


#### 과제 1 해설

실행 결과 (2026-11-01)

| 구분 | 노드 | 레이블 | 관계 |
|---|---|---|---|
| 허용 목록 있음 | 13개 | Award, Country, Field, Organization, Person (5종) | CITIZEN_OF, MARRIED_TO, RESEARCHED, WON, WORKED_AT (5종) |
| 허용 목록 없음 | 16개 | Award, Date, Family, Field, Nationality, Organization, Person, Profession (8종) | BIRTH_DATE, CO-WINNER, DEATH_DATE, HAS_NATIONALITY, HAS_PROFESSION, LEGACY, PROFESSOR, RESEARCHED, WINNER, WON, WON_IN_TWO, WON_TWICE (12종) |

- 허용 목록이 없으면 생몰일(`BIRTH_DATE`, `DEATH_DATE`), 교수 임용(`PROFESSOR`)처럼 더 많은 정보를 잡습니다.
- 대신 이름이 제각각이 됩니다. "상을 받았다"는 같은 뜻이 `WON`, `WINNER`, `WON_TWICE`, `WON_IN_TWO` 네 가지로 나뉘었습니다. 8장처럼 Cypher로 조회할 때 어떤 이름을 써야 할지 알 수 없고, 실행할 때마다 이름이 또 바뀝니다.
- `CO-WINNER`처럼 하이픈이 들어간 관계 이름은 Cypher에서 `` `CO-WINNER` ``처럼 백틱으로 감싸야 합니다.
- 허용 목록은 일관성을 얻는 대신 목록에 없는 정보를 잃습니다. 필요한 관계(예: `PROFESSOR_AT`, `SIBLING_OF`)를 미리 목록에 넣어 설계합니다.

### 과제 2. Pierre Curie 문장 추가

Pierre Curie의 경력 문장 4개를 같은 방법(같은 `transformer`)으로 추출해 적재하고, Pierre Curie 노드가 어떤 노드와 연결되는지 확인하세요. 새 청크도 벡터 검색에 쓰이도록 임베딩까지 합니다.

In [10]:
# 과제 2 정답
pierre_text = """
Pierre Curie was a French physicist who worked at the University of Paris.
He discovered piezoelectricity together with his brother Jacques Curie.
Pierre Curie shared the 1903 Nobel Prize in Physics with Marie Curie and Henri Becquerel.
He died in a street accident in Paris in 1906.
"""
# 1~3절의 splitter와 transformer를 그대로 쓴다. 허용 목록이 같으므로 기존 그래프와 이름 규칙이 맞는다.
pierre_documents = splitter.create_documents([pierre_text])
pierre_graph = transformer.convert_to_graph_documents(pierre_documents)
graph.add_graph_documents(pierre_graph, baseEntityLabel=True, include_source=True)

# 새 Document 노드에는 아직 임베딩이 없다. 같은 함수를 다시 부르면 임베딩이 없는 노드만 임베딩한다.
Neo4jVector.from_existing_graph(
    embedding=embeddings, **NEO4J,
    index_name="document_index", node_label="Document",
    text_node_properties=["text"], embedding_node_property="embedding",
)

# Marie Curie 문서에서 만든 Pierre Curie와 새 문서의 Pierre Curie는 id가 같아 한 노드로 합쳐진다.
# 그래서 두 문서의 관계가 한 노드에 모인다. 벡터 검색만으로는 얻기 어려운 연결이다.
print("[Pierre Curie의 관계]")
for row in graph.query("""
MATCH (p:__Entity__ {id: "Pierre Curie"})-[r]-(x)
RETURN type(r) AS rel, startNode(r) = p AS outgoing, [l IN labels(x) WHERE NOT l STARTS WITH '__'][0] AS label,
       coalesce(x.id, left(x.text, 50)) AS other
ORDER BY rel
"""):
    print(row)

# 표기가 다르면(예: 'Pierre' 와 'Pierre Curie') 다른 노드가 된다. 엔티티 중복 문제를 보여 주는 지점이다.
print("\n[이름에 Curie가 들어간 엔티티]")
print(graph.query("MATCH (n:__Entity__) WHERE n.id CONTAINS 'Curie' RETURN n.id AS id, count(*) AS nodes"))

[Pierre Curie의 관계]
{'rel': 'CITIZEN_OF', 'outgoing': True, 'label': 'Country', 'other': 'France'}
{'rel': 'MARRIED_TO', 'outgoing': True, 'label': 'Organization', 'other': 'Curie Family'}
{'rel': 'MENTIONS', 'outgoing': False, 'label': 'Document', 'other': 'dc9530f4c4bf8cdf78ccdffdadff8517'}
{'rel': 'MENTIONS', 'outgoing': False, 'label': 'Document', 'other': '781f66bba857a597b4a4a1578fa5dc7a'}
{'rel': 'RESEARCHED', 'outgoing': True, 'label': 'Person', 'other': 'Jacques Curie'}
{'rel': 'RESEARCHED', 'outgoing': True, 'label': 'Field', 'other': 'Piezoelectricity'}
{'rel': 'WON', 'outgoing': True, 'label': 'Award', 'other': '1903 Nobel Prize In Physics'}
{'rel': 'WON', 'outgoing': True, 'label': 'Award', 'other': 'Nobel Prize'}
{'rel': 'WORKED_AT', 'outgoing': True, 'label': 'Organization', 'other': 'University Of Paris'}

[이름에 Curie가 들어간 엔티티]
[{'id': 'Curie Family', 'nodes': 1}, {'id': 'Jacques Curie', 'nodes': 1}, {'id': 'Marie Curie', 'nodes': 1}, {'id': 'Pierre Curie', 'nodes': 1}]


#### 과제 2 해설

실행 결과 (2026-11-01): Pierre Curie 노드는 하나이고, 다음과 연결됐습니다.

| 관계 | 연결된 노드 | 평가 |
|---|---|---|
| WORKED_AT | University Of Paris | 맞음 |
| CITIZEN_OF | France | 맞음 |
| RESEARCHED | Piezoelectricity | 맞음 |
| WON | 1903 Nobel Prize In Physics, Nobel Prize | 같은 상이 노드 두 개로 나뉨 |
| MARRIED_TO | Curie Family (Organization) | 틀림. 배우자는 Marie Curie여야 함 |
| RESEARCHED | Jacques Curie (Person) | 틀림. 형제 관계가 허용 목록에 없어 가장 가까운 이름에 억지로 넣음 |
| MENTIONS (들어옴) | Document 2개 | 기존 텍스트와 새 텍스트 양쪽에서 언급됨 |

- 같은 대상이 하나의 노드로 합쳐진 경우: "Pierre Curie"는 기존 텍스트와 새 텍스트에서 같은 이름으로 추출돼 `MERGE`로 합쳐졌습니다. "University of Paris"도 하나입니다. LangChain은 노드 id의 단어 첫 글자를 대문자로 바꿔(`University Of Paris`) 표기를 통일합니다.
- 합쳐지지 않은 경우: "1903 Nobel Prize In Physics"와 "Nobel Prize"는 같은 상인데 이름이 달라 별개 노드가 됐습니다. 이름만으로는 같은 대상인지 판단할 수 없습니다.
- 3번 셀 결과에도 같은 문제가 있습니다. 두 번째 청크 "She was the first woman to win a Nobel Prize..."에서 "She"가 Person 노드로 추출됐습니다. 청크를 300자로 나눠서 앞 청크의 "Marie Curie"를 볼 수 없었기 때문입니다. 그래서 "노벨상을 두 번 받았다"는 정보가 Marie Curie가 아니라 "She"에 붙었습니다.
- 새 청크를 적재한 뒤 `from_existing_graph`를 다시 불러 새 Document에도 임베딩을 넣었습니다(Document 4개 모두 임베딩됨). 이 단계를 빠뜨리면 새 청크는 벡터 검색에 나오지 않습니다.
- 정리하면 지식 그래프의 품질은 엔티티 정규화(같은 대상을 하나로), 대명사 해소(She → Marie Curie), 관계 목록 설계에 달려 있습니다.

### 과제 3. k를 1과 3으로 바꿔 비교

같은 질문을 `k=1`과 `k=3`인 retriever로 실행하고, 검색된 청크와 답변을 비교하세요. 질문은 "마리 퀴리가 파리 대학에서 맡은 일은 무엇인가요?"를 씁니다.

In [11]:
# 과제 3 정답
# 답이 한 청크에만 있는 질문(1906년 파리 대학 교수)을 고른다.
question = "마리 퀴리가 파리 대학에서 맡은 일은 무엇인가요?"

for k in (1, 3):
    # k만 바꾼 retriever와 체인을 새로 만든다. 프롬프트와 LLM은 5절과 같다.
    k_retriever = vector_store.as_retriever(search_kwargs={"k": k})
    k_chain = (
        {"context": k_retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print(f"== k={k}")
    # 무엇이 검색됐는지 함께 봐야 답변이 달라진 원인을 설명할 수 있다.
    for doc in k_retriever.invoke(question):
        print("  검색:", doc.page_content.replace("\n", " ")[:90])
    print("  답변:", k_chain.invoke(question))

== k=1
  검색:  text: Pierre Curie was a French physicist who worked at the University of Paris. He disco
  답변: 정보가 부족합니다.
== k=3
  검색:  text: Pierre Curie was a French physicist who worked at the University of Paris. He disco
  검색:  text: Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them th
  검색:  text: Marie Curie, 7 November 1867 - 4 July 1934, was a Polish and naturalised-French phy
  답변: 마리 퀴리는 1906년에 파리 대학에서 최초의 여성 교수가 되었습니다.


#### 과제 3 해설

실행 결과 (2026-11-01)

| k | 검색된 청크 | 답변 |
|---|---|---|
| 1 | Pierre Curie 청크("worked at the University of Paris...") | 정보가 부족합니다. |
| 3 | Pierre Curie 청크, 남편·1906년 청크, 마리 퀴리 소개 청크 | 마리 퀴리는 1906년에 파리 대학에서 최초의 여성 교수가 되었습니다. |

- `k=1`일 때는 질문의 "파리 대학"과 단어가 겹치는 Pierre Curie 청크가 1위로 검색됐고, 답이 들어 있는 청크("She was, in 1906, the first woman to become a professor...")는 빠졌습니다. 프롬프트가 "문서에 없으면 정보가 부족하다고 답하라"고 했으므로 LLM은 정직하게 모른다고 답했습니다.
- `k=3`이면 답이 든 청크가 포함되어 맞게 답했습니다.
- 과제 2에서 Pierre Curie 청크를 추가하기 전이라면 1위 청크가 달랐을 것입니다. 문서가 늘면 같은 질문의 검색 결과도 바뀝니다.
- `k`는 정확도와 비용 사이의 조절값입니다. 크게 하면 답을 놓칠 위험이 줄지만, 프롬프트가 길어지고 관련 없는 내용이 섞입니다.